In [10]:
!pip uninstall -y numpy
!pip install numpy==1.24.3
!pip install --upgrade --force-reinstall PyMuPDF Pillow

Found existing installation: numpy 2.2.6
Uninstalling numpy-2.2.6:
  Successfully uninstalled numpy-2.2.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 121.9 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Not uninstalling numpy at /toolkit-cache/1.1.5/python3.10/kernel-libs/lib/python3.10/site-packages, outside environment /root/venv
    Can't uninstall 'numpy'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy 3.4.4 requires pydantic!=1.8,!=1.8.1,<1.11.0,>=1.7.4, but you have pydantic 2.12.5 which is incompatible.
spacy 3.4.4 requires typer<0.8.0,>=0.3.0, but you have typer 0.19.2 which is incompatible.
deepnote-toolkit 1.1.5 requires pandas<2.2,>=1.2.5; python_version < "3.12", but you have pandas 2.3.3 which is incompatible.
  Using cached pymupdf-1.26.6-cp310-abi

In [13]:
#!/usr/bin/env python3
"""
8_GDP_REGRESSION_ANALYSIS.py

Analyzes relationship between GDP per capita and LLM prediction errors.
Similar to internet usage analysis but using GDP as the predictor.

Question: Do LLMs perform better on wealthier countries?
"""

# ================================================================
# Fix matplotlib compatibility
# ================================================================
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")

print("="*80)
print("GDP PER CAPITA REGRESSION ANALYSIS")
print("="*80)
print("\nAnalyzing relationship between GDP per capita and LLM prediction errors")

# ================================================================
# 1. Load Data
# ================================================================

print("\n" + "="*80)
print("1. LOADING DATA")
print("="*80)

# Load predictions (Stage 8 - most information)
df = pd.read_csv("predictions_all_stages_long.csv")
stage8_df = df[df['stage'] == 8].copy()

print(f"✓ Loaded predictions: {len(stage8_df)} rows (Stage 8)")

# Load ground truth with GDP data
gt_df = pd.read_csv("data_final.csv")

# Merge
stage8_country = stage8_df.groupby('countrynew').first().reset_index()
analysis_df = stage8_country.merge(
    gt_df[['countrynew', 'mean_other_willingness', 'gdp_capita_2021']], 
    on='countrynew', 
    how='inner'
)

print(f"✓ Merged data: {len(analysis_df)} countries")

# ================================================================
# 2. Calculate Prediction Errors (MAE per country)
# ================================================================

print("\n" + "="*80)
print("2. CALCULATING PREDICTION ERRORS")
print("="*80)

# Ground truth (0-100 scale)
analysis_df['ground_truth'] = analysis_df['mean_other_willingness'] * 100

# Calculate absolute errors for each model
models = ['gpt', 'claude', 'gemini', 'llama']

for model in models:
    pred_col = f'pred_{model}'
    error_col = f'error_{model}'
    
    # Absolute error
    analysis_df[error_col] = np.abs(analysis_df[pred_col] - analysis_df['ground_truth'])
    
    mean_error = analysis_df[error_col].mean()
    print(f"   {model.upper()}: Mean absolute error = {mean_error:.2f}pp")

# Calculate ensemble
analysis_df['pred_ensemble'] = analysis_df[['pred_gpt', 'pred_claude', 'pred_gemini', 'pred_llama']].mean(axis=1)
analysis_df['error_ensemble'] = np.abs(analysis_df['pred_ensemble'] - analysis_df['ground_truth'])

print(f"   ENSEMBLE: Mean absolute error = {analysis_df['error_ensemble'].mean():.2f}pp")

# ================================================================
# 3. Check GDP Data Availability
# ================================================================

print("\n" + "="*80)
print("3. GDP PER CAPITA DATA")
print("="*80)

# Check for missing GDP data
n_total = len(analysis_df)
n_with_gdp = analysis_df['gdp_capita_2021'].notna().sum()
n_missing_gdp = n_total - n_with_gdp

print(f"\nCountries with GDP data: {n_with_gdp}/{n_total}")
print(f"Countries missing GDP data: {n_missing_gdp}/{n_total}")

if n_missing_gdp > 0:
    print(f"\nCountries without GDP data:")
    missing = analysis_df[analysis_df['gdp_capita_2021'].isna()]['countrynew'].tolist()
    for country in missing[:10]:  # Show first 10
        print(f"   - {country}")
    if len(missing) > 10:
        print(f"   ... and {len(missing) - 10} more")

# Remove countries without GDP data
analysis_df = analysis_df[analysis_df['gdp_capita_2021'].notna()].copy()

print(f"\n✓ Final dataset: {len(analysis_df)} countries with complete data")

# GDP statistics
print(f"\nGDP per capita statistics:")
print(f"   Min: ${analysis_df['gdp_capita_2021'].min():,.0f}")
print(f"   Max: ${analysis_df['gdp_capita_2021'].max():,.0f}")
print(f"   Mean: ${analysis_df['gdp_capita_2021'].mean():,.0f}")
print(f"   Median: ${analysis_df['gdp_capita_2021'].median():,.0f}")

# ================================================================
# 4. Correlation Analysis
# ================================================================

print("\n" + "="*80)
print("4. CORRELATION ANALYSIS")
print("="*80)

print("\nCorrelation between GDP per capita and prediction error:\n")

correlation_results = []

for model in models + ['ensemble']:
    error_col = f'error_{model}'
    
    # Pearson correlation
    r_pearson, p_pearson = pearsonr(analysis_df['gdp_capita_2021'], analysis_df[error_col])
    
    # Spearman correlation (rank-based, robust to outliers)
    r_spearman, p_spearman = spearmanr(analysis_df['gdp_capita_2021'], analysis_df[error_col])
    
    # Significance stars
    sig_pearson = ""
    if p_pearson < 0.001:
        sig_pearson = "***"
    elif p_pearson < 0.01:
        sig_pearson = "**"
    elif p_pearson < 0.05:
        sig_pearson = "*"
    
    sig_spearman = ""
    if p_spearman < 0.001:
        sig_spearman = "***"
    elif p_spearman < 0.01:
        sig_spearman = "**"
    elif p_spearman < 0.05:
        sig_spearman = "*"
    
    print(f"{model.upper():12} | Pearson r = {r_pearson:+.3f}{sig_pearson:3} (p={p_pearson:.4f}) | Spearman ρ = {r_spearman:+.3f}{sig_spearman:3} (p={p_spearman:.4f})")
    
    correlation_results.append({
        'Model': model.upper(),
        'Pearson r': r_pearson,
        'Pearson p': p_pearson,
        'Spearman ρ': r_spearman,
        'Spearman p': p_spearman
    })

print("\n* p < .05, ** p < .01, *** p < .001")
print("\nNegative correlation = Lower error for higher GDP (better performance on wealthy countries)")
print("Positive correlation = Higher error for higher GDP (worse performance on wealthy countries)")

# Save correlation results
corr_df = pd.DataFrame(correlation_results)
corr_df.to_csv('gdp_correlation_results.csv', index=False)
print("\n✓ Saved gdp_correlation_results.csv")

# ================================================================
# 5. Linear Regression Analysis
# ================================================================

print("\n" + "="*80)
print("5. LINEAR REGRESSION ANALYSIS")
print("="*80)

print("\nRegression: Error = β₀ + β₁ × GDP_per_capita\n")

regression_results = []

for model in models + ['ensemble']:
    error_col = f'error_{model}'
    
    # Prepare data
    X = analysis_df['gdp_capita_2021'].values.reshape(-1, 1)
    y = analysis_df[error_col].values
    
    # Fit linear regression
    reg = LinearRegression()
    reg.fit(X, y)
    
    intercept = reg.intercept_
    slope = reg.coef_[0]
    
    # R-squared
    r2 = reg.score(X, y)
    
    # Predict
    y_pred = reg.predict(X)
    
    # Calculate what this means practically
    error_at_1k = intercept + slope * 1000
    error_at_10k = intercept + slope * 10000
    error_at_50k = intercept + slope * 50000
    error_at_100k = intercept + slope * 100000
    
    print(f"{model.upper()}:")
    print(f"   Intercept (β₀): {intercept:.2f}pp")
    print(f"   Slope (β₁): {slope:.6f}pp per $1")
    print(f"   R²: {r2:.4f}")
    print(f"   Predicted error at GDP=$1,000: {error_at_1k:.2f}pp")
    print(f"   Predicted error at GDP=$10,000: {error_at_10k:.2f}pp")
    print(f"   Predicted error at GDP=$50,000: {error_at_50k:.2f}pp")
    print(f"   Predicted error at GDP=$100,000: {error_at_100k:.2f}pp")
    print()
    
    regression_results.append({
        'Model': model.upper(),
        'Intercept': intercept,
        'Slope': slope,
        'R²': r2,
        'Error_GDP_1k': error_at_1k,
        'Error_GDP_10k': error_at_10k,
        'Error_GDP_50k': error_at_50k,
        'Error_GDP_100k': error_at_100k
    })

# Save regression results
reg_df = pd.DataFrame(regression_results)
reg_df.to_csv('gdp_regression_results.csv', index=False)
print("✓ Saved gdp_regression_results.csv")

# ================================================================
# 6. Visualizations
# ================================================================

print("\n" + "="*80)
print("6. CREATING VISUALIZATIONS")
print("="*80)

# Figure 1: Scatter plots with regression lines (2x2 grid for 4 models)
fig = plt.figure(figsize=(14, 10))
fig.suptitle('LLM Prediction Error vs GDP per Capita', fontsize=16, fontweight='bold')

axes = []
for i in range(2):
    row_axes = []
    for j in range(2):
        ax = fig.add_subplot(2, 2, i*2 + j + 1)
        row_axes.append(ax)
    axes.append(row_axes)

for idx, model in enumerate(models):
    row = idx // 2
    col = idx % 2
    ax = axes[row][col]
    
    error_col = f'error_{model}'
    
    # Scatter plot
    ax.scatter(analysis_df['gdp_capita_2021'], analysis_df[error_col], 
               alpha=0.6, s=50, color='steelblue', edgecolors='black', linewidth=0.5)
    
    # Regression line
    X = analysis_df['gdp_capita_2021'].values.reshape(-1, 1)
    y = analysis_df[error_col].values
    reg = LinearRegression()
    reg.fit(X, y)
    
    x_line = np.linspace(analysis_df['gdp_capita_2021'].min(), 
                         analysis_df['gdp_capita_2021'].max(), 100)
    y_line = reg.predict(x_line.reshape(-1, 1))
    
    ax.plot(x_line, y_line, 'r--', linewidth=2, label='Regression line')
    
    # Get correlation
    r_pearson = corr_df[corr_df['Model'] == model.upper()]['Pearson r'].values[0]
    p_pearson = corr_df[corr_df['Model'] == model.upper()]['Pearson p'].values[0]
    
    sig = ""
    if p_pearson < 0.001:
        sig = "***"
    elif p_pearson < 0.01:
        sig = "**"
    elif p_pearson < 0.05:
        sig = "*"
    
    # Labels
    ax.set_xlabel('GDP per Capita (USD)', fontweight='bold', fontsize=11)
    ax.set_ylabel('Prediction Error (pp)', fontweight='bold', fontsize=11)
    ax.set_title(f'{model.upper()}\nr = {r_pearson:.3f}{sig}, n = {len(analysis_df)}', 
                 fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=9)
    
    # Format x-axis as currency
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}k'))

plt.tight_layout()
plt.savefig('gdp_vs_error_scatter.png', dpi=300, bbox_inches='tight')
plt.savefig('gdp_vs_error_scatter.pdf', dpi=300, bbox_inches='tight')
print("✓ Saved gdp_vs_error_scatter.png")
print("✓ Saved gdp_vs_error_scatter.pdf")
plt.close()

# Figure 2: Correlation comparison (only 4 models)
fig, ax = plt.subplots(figsize=(10, 6))

models_viz = models  # Only 4 models, no ensemble
x_pos = np.arange(len(models_viz))
correlations = [corr_df[corr_df['Model'] == m.upper()]['Pearson r'].values[0] for m in models_viz]
p_values = [corr_df[corr_df['Model'] == m.upper()]['Pearson p'].values[0] for m in models_viz]

# Color bars by significance
colors = []
for p in p_values:
    if p < 0.001:
        colors.append('#d62728')  # Dark red - highly significant
    elif p < 0.01:
        colors.append('#ff7f0e')  # Orange - significant
    elif p < 0.05:
        colors.append('#ffbb78')  # Light orange - marginally significant
    else:
        colors.append('#bdbdbd')  # Gray - not significant

bars = ax.bar(x_pos, correlations, alpha=0.8, color=colors, edgecolor='black', linewidth=1)

ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.set_ylabel('Pearson Correlation (r)', fontweight='bold', fontsize=12)
ax.set_xlabel('Model', fontweight='bold', fontsize=12)
ax.set_title('Correlation: GDP per Capita vs Prediction Error', fontweight='bold', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels([m.upper() for m in models_viz], fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Add significance stars on bars
for i, (bar, corr, p) in enumerate(zip(bars, correlations, p_values)):
    sig = ""
    if p < 0.001:
        sig = "***"
    elif p < 0.01:
        sig = "**"
    elif p < 0.05:
        sig = "*"
    
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01 if height > 0 else height - 0.01,
            f'{corr:.3f}{sig}', ha='center', va='bottom' if height > 0 else 'top', 
            fontweight='bold', fontsize=10)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#d62728', label='p < .001 ***'),
    Patch(facecolor='#ff7f0e', label='p < .01 **'),
    Patch(facecolor='#ffbb78', label='p < .05 *'),
    Patch(facecolor='#bdbdbd', label='n.s.')
]
ax.legend(handles=legend_elements, loc='best', fontsize=10)

ax.text(0.02, 0.98, 'Negative correlation = Better performance on wealthy countries\nPositive correlation = Worse performance on wealthy countries', 
        transform=ax.transAxes, fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig('gdp_correlation_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig('gdp_correlation_comparison.pdf', dpi=300, bbox_inches='tight')
print("✓ Saved gdp_correlation_comparison.png")
print("✓ Saved gdp_correlation_comparison.pdf")
plt.close()

# Figure 3: Error by GDP quintiles (2x2 grid for 4 models)
fig = plt.figure(figsize=(14, 10))
fig.suptitle('Prediction Error by GDP Quintile', fontsize=16, fontweight='bold')

axes = []
for i in range(2):
    row_axes = []
    for j in range(2):
        ax = fig.add_subplot(2, 2, i*2 + j + 1)
        row_axes.append(ax)
    axes.append(row_axes)

# Create GDP quintiles
analysis_df['gdp_quintile'] = pd.qcut(analysis_df['gdp_capita_2021'], 
                                       q=5, 
                                       labels=['Q1\n(Poorest)', 'Q2', 'Q3', 'Q4', 'Q5\n(Richest)'])

for idx, model in enumerate(models):
    row = idx // 2
    col = idx % 2
    ax = axes[row][col]
    
    error_col = f'error_{model}'
    
    # Box plot by quintile
    quintile_data = [analysis_df[analysis_df['gdp_quintile'] == q][error_col].values 
                     for q in ['Q1\n(Poorest)', 'Q2', 'Q3', 'Q4', 'Q5\n(Richest)']]
    
    bp = ax.boxplot(quintile_data, labels=['Q1\n(Poorest)', 'Q2', 'Q3', 'Q4', 'Q5\n(Richest)'], 
                    patch_artist=True)
    
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    
    ax.set_ylabel('Prediction Error (pp)', fontweight='bold', fontsize=11)
    ax.set_xlabel('GDP Quintile', fontweight='bold', fontsize=11)
    ax.set_title(f'{model.upper()}', fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('gdp_quintile_boxplots.png', dpi=300, bbox_inches='tight')
plt.savefig('gdp_quintile_boxplots.pdf', dpi=300, bbox_inches='tight')
print("✓ Saved gdp_quintile_boxplots.png")
print("✓ Saved gdp_quintile_boxplots.pdf")
plt.close()

# ================================================================
# 7. Summary Statistics
# ================================================================

print("\n" + "="*80)
print("7. SUMMARY STATISTICS")
print("="*80)

# Error by GDP quintile
print("\nMean prediction error by GDP quintile:\n")

quintile_summary = []

for model in models_viz:
    error_col = f'error_{model}'
    
    q1_mean = analysis_df[analysis_df['gdp_quintile'] == 'Q1\n(Poorest)'][error_col].mean()
    q5_mean = analysis_df[analysis_df['gdp_quintile'] == 'Q5\n(Richest)'][error_col].mean()
    difference = q5_mean - q1_mean
    
    quintile_summary.append({
        'Model': model.upper(),
        'Q1_Mean_Error': q1_mean,
        'Q5_Mean_Error': q5_mean,
        'Difference': difference
    })
    
    print(f"{model.upper():12} | Q1 (Poorest): {q1_mean:.2f}pp | Q5 (Richest): {q5_mean:.2f}pp | Diff: {difference:+.2f}pp")

print("\nPositive difference = Worse on wealthy countries")
print("Negative difference = Better on wealthy countries")

# Save quintile summary
quintile_df = pd.DataFrame(quintile_summary)
quintile_df.to_csv('gdp_quintile_summary.csv', index=False)
print("\n✓ Saved gdp_quintile_summary.csv")

# ================================================================
# 8. Export Full Dataset
# ================================================================

print("\n" + "="*80)
print("8. EXPORTING DATASET")
print("="*80)

# Save full analysis dataset
export_cols = ['countrynew', 'gdp_capita_2021', 'ground_truth', 
               'pred_gpt', 'pred_claude', 'pred_gemini', 'pred_llama', 'pred_ensemble',
               'error_gpt', 'error_claude', 'error_gemini', 'error_llama', 'error_ensemble',
               'gdp_quintile']

analysis_df[export_cols].to_csv('gdp_analysis_dataset.csv', index=False)
print("✓ Saved gdp_analysis_dataset.csv")

# ================================================================
# 9. Final Summary
# ================================================================

print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"\nDataset: {len(analysis_df)} countries with complete GDP data")
print(f"GDP range: ${analysis_df['gdp_capita_2021'].min():,.0f} - ${analysis_df['gdp_capita_2021'].max():,.0f}")

print("\n" + "="*60)
print("CORRELATION SUMMARY")
print("="*60)

print("\nStrongest correlations (absolute value):")
corr_sorted = corr_df.sort_values('Pearson r', key=abs, ascending=False)
for _, row in corr_sorted.iterrows():
    sig = ""
    if row['Pearson p'] < 0.001:
        sig = "***"
    elif row['Pearson p'] < 0.01:
        sig = "**"
    elif row['Pearson p'] < 0.05:
        sig = "*"
    
    direction = "Better on wealthy" if row['Pearson r'] < 0 else "Worse on wealthy"
    print(f"   {row['Model']:12} | r = {row['Pearson r']:+.3f}{sig:3} | {direction}")

print("\n" + "="*60)
print("FILES CREATED")
print("="*60)

print("\nVisualization files:")
print("   - gdp_vs_error_scatter.png / .pdf (scatter plots with regression lines)")
print("   - gdp_correlation_comparison.png / .pdf (correlation bar chart)")
print("   - gdp_quintile_boxplots.png / .pdf (error by GDP quintile)")

print("\nData files:")
print("   - gdp_correlation_results.csv (correlation statistics)")
print("   - gdp_regression_results.csv (regression coefficients)")
print("   - gdp_quintile_summary.csv (error by quintile)")
print("   - gdp_analysis_dataset.csv (full dataset)")

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/local/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/local/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/toolkit-cache/1.1.5/python3.10/kernel-libs/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/root/venv/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/toolkit-cache/1

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [1]:
#!/usr/bin/env python3
"""
8_GDP_REGRESSION_INTERNET_FORMAT.py

Analyzes the relationship between prediction error (MAE) and GDP per capita.

Structure mirrors 6_INTERNET_USAGE_REGRESSION.py:
1. Aggregate (across all stages): MAE vs GDP per capita
2. By stage: MAE at each stage vs GDP per capita
3. Visualisations with country labels and heatmaps

Additional: includes log GDP per capita as an extra aggregate scatter plot.

Question: Do LLMs perform better on wealthier countries?
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Try to import adjustText for better label positioning
try:
    from adjustText import adjust_text
    HAS_ADJUST_TEXT = True
except ImportError:
    HAS_ADJUST_TEXT = False
    print("⚠️  adjustText not installed - country labels may overlap")
    print("   Install with: pip install adjustText")

# Set style
sns.set_style("whitegrid")
# Set default figure size - will be overridden by individual figures
import matplotlib
matplotlib.use('Agg')

print("="*80)
print("GDP PER CAPITA vs PREDICTION ERROR ANALYSIS")
print("="*80)

# ================================================================
# 1. Load Data
# ================================================================

print("\n" + "="*80)
print("1. LOADING DATA")
print("="*80)

# Load predictions (all stages)
df = pd.read_csv("predictions_all_stages_long.csv")
print(f"✓ Loaded predictions: {len(df)} rows")

# Load ground truth and GDP per capita
gt_df = pd.read_csv("data_final.csv")
# Ground truth in percentage points (0-100)
gt_df['ground_truth_pi'] = gt_df['mean_other_willingness'] * 100

# Merge ground truth & GDP into prediction data
df = df.merge(
    gt_df[['countrynew', 'ground_truth_pi', 'gdp_capita_2021']],
    on='countrynew',
    how='left'
)
print("✓ Merged ground truth and GDP")

# Remove rows with missing data (ground truth or GDP)
df_complete = df[df['ground_truth_pi'].notna() & df['gdp_capita_2021'].notna()].copy()
print(f"\n✓ Final dataset: {len(df_complete)} rows")
print(f"   Countries: {df_complete['countrynew'].nunique()}")
print(f"   GDP per capita range: ${df_complete['gdp_capita_2021'].min():,.0f} - ${df_complete['gdp_capita_2021'].max():,.0f}")

# Create log GDP per capita
df_complete['log_gdp_capita_2021'] = np.log(df_complete['gdp_capita_2021'])
print("\nCreated log GDP per capita (natural log).")
print(f"   Mean log GDP: {df_complete['log_gdp_capita_2021'].mean():.2f}")

# ================================================================
# 2. Calculate MAE for each model
# ================================================================

print("\n" + "="*80)
print("2. CALCULATING MAE")
print("="*80)

models = ['gpt', 'claude', 'gemini', 'llama']

for model in models:
    pred_col = f'pred_{model}'
    mae_col = f'mae_{model}'
    df_complete[mae_col] = abs(df_complete[pred_col] - df_complete['ground_truth_pi'])

print("✓ Calculated MAE for all models")

# ================================================================
# 3. Aggregate Analysis (across all stages)
# ================================================================

print("\n" + "="*80)
print("3. AGGREGATE REGRESSION (Across All Stages)")
print("="*80)

# Aggregate by country: mean MAE and GDP per capita (and log GDP)
country_aggregate = df_complete.groupby('countrynew').agg({
    'mae_gpt': 'mean',
    'mae_claude': 'mean',
    'mae_gemini': 'mean',
    'mae_llama': 'mean',
    'gdp_capita_2021': 'first',
    'log_gdp_capita_2021': 'first'
}).reset_index()

print(f"\n✓ Aggregated data: {len(country_aggregate)} countries")
print(f"   GDP per capita (min/median/max): ${country_aggregate['gdp_capita_2021'].min():,.0f} / ${country_aggregate['gdp_capita_2021'].median():,.0f} / ${country_aggregate['gdp_capita_2021'].max():,.0f}")

# Run regressions for each model (MAE vs GDP per capita)
aggregate_results = []

for model in models:
    mae_col = f'mae_{model}'

    data = country_aggregate[[mae_col, 'gdp_capita_2021']].dropna()

    if len(data) == 0:
        print(f"\n⚠️  No data for {model.upper()}")
        continue

    X = data['gdp_capita_2021'].values.reshape(-1, 1)
    y = data[mae_col].values

    lr = LinearRegression()
    lr.fit(X, y)

    y_pred = lr.predict(X)
    r2 = r2_score(y, y_pred)

    # Slope per $10,000 GDP per capita to keep numbers interpretable
    base_slope = lr.coef_[0]  # MAE change per $1
    slope_10k = base_slope * 10000  # MAE change per $10k

    # Pearson correlation
    pearson_r, pearson_p = pearsonr(data['gdp_capita_2021'], data[mae_col])

    # Spearman correlation
    spearman_r, spearman_p = spearmanr(data['gdp_capita_2021'], data[mae_col])

    print(f"\n{model.upper()}:")
    print(f"   N countries: {len(data)}")
    print(f"   Slope: {slope_10k:.4f} (MAE change per $10k GDP per capita)")
    print(f"   Intercept: {lr.intercept_:.2f}")
    print(f"   R²: {r2:.4f}")
    print(f"   Pearson r: {pearson_r:.4f}, p = {pearson_p:.4f}")
    print(f"   Spearman ρ: {spearman_r:.4f}, p = {spearman_p:.4f}")

    aggregate_results.append({
        'Model': model.upper(),
        'N': len(data),
        'Slope_per_10k': slope_10k,
        'Intercept': lr.intercept_,
        'R²': r2,
        'Pearson r': pearson_r,
        'Pearson p': pearson_p,
        'Spearman ρ': spearman_r,
        'Spearman p': spearman_p
    })

aggregate_df = pd.DataFrame(aggregate_results)

print("\n" + "="*60)
print("AGGREGATE RESULTS SUMMARY:")
print("="*60)
print(aggregate_df.to_string(index=False))

# ================================================================
# 4. By-Stage Analysis
# ================================================================

print("\n" + "="*80)
print("4. BY-STAGE REGRESSION")
print("="*80)

stage_results = []

for stage in sorted(df_complete['stage'].unique()):
    print(f"\n{'='*60}")
    print(f"Stage {stage}:")
    print('='*60)

    stage_data = df_complete[df_complete['stage'] == stage].copy()

    # Aggregate by country for this stage
    stage_country = stage_data.groupby('countrynew').agg({
        'mae_gpt': 'mean',
        'mae_claude': 'mean',
        'mae_gemini': 'mean',
        'mae_llama': 'mean',
        'gdp_capita_2021': 'first'
    }).reset_index()

    for model in models:
        mae_col = f'mae_{model}'

        data = stage_country[[mae_col, 'gdp_capita_2021']].dropna()

        if len(data) == 0:
            continue

        X = data['gdp_capita_2021'].values.reshape(-1, 1)
        y = data[mae_col].values

        lr = LinearRegression()
        lr.fit(X, y)

        y_pred = lr.predict(X)
        r2 = r2_score(y, y_pred)

        base_slope = lr.coef_[0]
        slope_10k = base_slope * 10000

        pearson_r, pearson_p = pearsonr(data['gdp_capita_2021'], data[mae_col])
        spearman_r, spearman_p = spearmanr(data['gdp_capita_2021'], data[mae_col])

        print(f"\n   {model.upper()}:")
        print(f"      Slope: {slope_10k:.4f} (per $10k), R²: {r2:.4f}, r: {pearson_r:.4f} (p={pearson_p:.4f})")

        stage_results.append({
            'Stage': stage,
            'Model': model.upper(),
            'N': len(data),
            'Slope_per_10k': slope_10k,
            'Intercept': lr.intercept_,
            'R²': r2,
            'Pearson r': pearson_r,
            'Pearson p': pearson_p,
            'Spearman ρ': spearman_r,
            'Spearman p': spearman_p
        })

stage_df = pd.DataFrame(stage_results)

# ================================================================
# 5. Country Code Mapping
# ================================================================

print("\n" + "="*80)
print("5. CREATING COUNTRY CODE MAPPING")
print("="*80)

# ISO 3-letter country codes (same mapping as internet script)
country_codes = {
    'Afghanistan': 'AFG', 'Albania': 'ALB', 'Algeria': 'DZA', 'Argentina': 'ARG',
    'Armenia': 'ARM', 'Australia': 'AUS', 'Austria': 'AUT', 'Bangladesh': 'BGD',
    'Belgium': 'BEL', 'Benin': 'BEN', 'Bolivia': 'BOL', 'Bosnia Herzegovina': 'BIH',
    'Botswana': 'BWA', 'Brazil': 'BRA', 'Bulgaria': 'BGR', 'Burkina Faso': 'BFA',
    'Cambodia': 'KHM', 'Cameroon': 'CMR', 'Canada': 'CAN', 'Chad': 'TCD',
    'Chile': 'CHL', 'China': 'CHN', 'Colombia': 'COL', 'Congo Brazzaville': 'COG',
    'Costa Rica': 'CRI', 'Croatia': 'HRV', 'Cyprus': 'CYP', 'Czech Republic': 'CZE',
    'Denmark': 'DNK', 'Dominican Republic': 'DOM', 'Ecuador': 'ECU', 'Egypt': 'EGY',
    'El Salvador': 'SLV', 'Estonia': 'EST', 'Ethiopia': 'ETH', 'Finland': 'FIN',
    'France': 'FRA', 'Gabon': 'GAB', 'Georgia': 'GEO', 'Germany': 'DEU',
    'Ghana': 'GHA', 'Greece': 'GRC', 'Guatemala': 'GTM', 'Guinea': 'GIN',
    'Haiti': 'HTI', 'Honduras': 'HND', 'Hong Kong': 'HKG', 'Hungary': 'HUN',
    'Iceland': 'ISL', 'India': 'IND', 'Indonesia': 'IDN', 'Iran': 'IRN',
    'Iraq': 'IRQ', 'Ireland': 'IRL', 'Israel': 'ISR', 'Italy': 'ITA',
    'Ivory Coast': 'CIV', 'Jamaica': 'JAM', 'Japan': 'JPN', 'Jordan': 'JOR',
    'Kazakhstan': 'KAZ', 'Kenya': 'KEN', 'Kosovo': 'XKX', 'Kyrgyzstan': 'KGZ',
    'Laos': 'LAO', 'Latvia': 'LVA', 'Lebanon': 'LBN', 'Liberia': 'LBR', 'Libya': 'LBY',
    'Lithuania': 'LTU', 'Luxembourg': 'LUX', 'Macedonia': 'MKD', 'Madagascar': 'MDG',
    'Malawi': 'MWI', 'Malaysia': 'MYS', 'Mali': 'MLI', 'Malta': 'MLT',
    'Mauritania': 'MRT', 'Mauritius': 'MUS', 'Mexico': 'MEX', 'Moldova': 'MDA',
    'Mongolia': 'MNG', 'Montenegro': 'MNE', 'Morocco': 'MAR', 'Mozambique': 'MOZ',
    'Myanmar': 'MMR', 'Namibia': 'NAM', 'Nepal': 'NPL', 'Netherlands': 'NLD',
    'New Zealand': 'NZL', 'Nicaragua': 'NIC', 'Niger': 'NER', 'Nigeria': 'NGA',
    'North Macedonia': 'MKD', 'Norway': 'NOR', 'Pakistan': 'PAK', 'Palestinian Territories': 'PSE',
    'Panama': 'PAN', 'Paraguay': 'PRY', 'Peru': 'PER', 'Philippines': 'PHL',
    'Poland': 'POL', 'Portugal': 'PRT', 'Romania': 'ROU', 'Russia': 'RUS', 'Rwanda': 'RWA',
    'Saudi Arabia': 'SAU', 'Senegal': 'SEN', 'Serbia': 'SRB', 'Sierra Leone': 'SLE',
    'Singapore': 'SGP', 'Slovakia': 'SVK', 'Slovenia': 'SVN', 'South Africa': 'ZAF',
    'South Korea': 'KOR', 'Spain': 'ESP', 'Sri Lanka': 'LKA', 'Sweden': 'SWE',
    'Switzerland': 'CHE', 'Taiwan': 'TWN', 'Tajikistan': 'TJK', 'Tanzania': 'TZA',
    'Thailand': 'THA', 'Togo': 'TGO', 'Tunisia': 'TUN', 'Turkey': 'TUR',
    'Turkmenistan': 'TKM', 'Uganda': 'UGA', 'Ukraine': 'UKR', 'United Arab Emirates': 'ARE',
    'United Kingdom': 'GBR', 'United States': 'USA', 'Uruguay': 'URY',
    'Uzbekistan': 'UZB', 'Venezuela': 'VEN', 'Vietnam': 'VNM', 'Yemen': 'YEM',
    'Zambia': 'ZMB', 'Zimbabwe': 'ZWE'
}

country_aggregate['country_code'] = country_aggregate['countrynew'].map(country_codes)
print(f"✓ Mapped {country_aggregate['country_code'].notna().sum()} country codes")

# ================================================================
# 6. Visualizations with Country Labels
# ================================================================

print("\n" + "="*80)
print("6. CREATING VISUALIZATIONS")
print("="*80)

# We only call adjust_text here if it was successfully imported above

# Figure 1: Aggregate scatter plots (4 models) with country labels (MAE vs GDP)
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('GDP per Capita vs Prediction Error (Aggregated Across All Stages)',
             fontsize=16, fontweight='bold')

for idx, model in enumerate(models):
    ax = axes[idx // 2, idx % 2]

    mae_col = f'mae_{model}'
    data = country_aggregate[['countrynew', 'country_code', mae_col, 'gdp_capita_2021']].dropna()

    # Scatter plot
    ax.scatter(data['gdp_capita_2021'], data[mae_col],
               alpha=0.6, s=50, c='steelblue', edgecolors='black', linewidth=0.5)

    # Country labels
    if HAS_ADJUST_TEXT:
        texts = []
        for _, row in data.iterrows():
            if pd.notna(row['country_code']):
                texts.append(ax.text(row['gdp_capita_2021'], row[mae_col],
                                     row['country_code'], fontsize=6, alpha=0.7))
        try:
            adjust_text(texts,
                        arrowprops=dict(arrowstyle='-', color='gray', lw=0.5, alpha=0.5),
                        ax=ax, expand_points=(1.2, 1.2), force_points=(0.5, 0.5))
        except Exception:
            pass

    # Regression line
    X = data['gdp_capita_2021'].values.reshape(-1, 1)
    y = data[mae_col].values
    lr = LinearRegression()
    lr.fit(X, y)

    x_line = np.linspace(data['gdp_capita_2021'].min(), data['gdp_capita_2021'].max(), 100)
    y_line = lr.predict(x_line.reshape(-1, 1))

    ax.plot(x_line, y_line, 'r--', linewidth=2, label='Linear fit')

    # Get stats
    r2 = aggregate_df[aggregate_df['Model'] == model.upper()]['R²'].values[0]
    pearson_r = aggregate_df[aggregate_df['Model'] == model.upper()]['Pearson r'].values[0]
    pearson_p = aggregate_df[aggregate_df['Model'] == model.upper()]['Pearson p'].values[0]
    slope_10k = aggregate_df[aggregate_df['Model'] == model.upper()]['Slope_per_10k'].values[0]

    sig = ""
    if pearson_p < 0.001:
        sig = "***"
    elif pearson_p < 0.01:
        sig = "**"
    elif pearson_p < 0.05:
        sig = "*"

    stats_text = f'r = {pearson_r:.3f}{sig}\nR² = {r2:.3f}\nSlope = {slope_10k:.4f} per $10k'
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
            verticalalignment='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax.set_xlabel('GDP per Capita (USD)', fontweight='bold', fontsize=11)
    ax.set_ylabel('Mean Absolute Error (pp)', fontweight='bold', fontsize=11)
    ax.set_title(f'{model.upper()}', fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}k'))

plt.tight_layout()
plt.savefig('gdp_aggregate_regression_with_labels.png', dpi=300, bbox_inches='tight')
plt.savefig('gdp_aggregate_regression_with_labels.pdf', dpi=300, bbox_inches='tight')
print("\n✓ Saved gdp_aggregate_regression_with_labels.png")
print("✓ Saved gdp_aggregate_regression_with_labels.pdf")
plt.close()

# Figure 1b: Aggregate scatter plots without labels (MAE vs GDP)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('GDP per Capita vs Prediction Error (Aggregated - No Labels)',
             fontsize=16, fontweight='bold')

for idx, model in enumerate(models):
    ax = axes[idx // 2, idx % 2]

    mae_col = f'mae_{model}'
    data = country_aggregate[[mae_col, 'gdp_capita_2021']].dropna()

    ax.scatter(data['gdp_capita_2021'], data[mae_col],
               alpha=0.6, s=60, c='steelblue', edgecolors='black', linewidth=0.5)

    X = data['gdp_capita_2021'].values.reshape(-1, 1)
    y = data[mae_col].values
    lr = LinearRegression()
    lr.fit(X, y)

    x_line = np.linspace(data['gdp_capita_2021'].min(), data['gdp_capita_2021'].max(), 100)
    y_line = lr.predict(x_line.reshape(-1, 1))

    ax.plot(x_line, y_line, 'r--', linewidth=2, label='Linear fit')

    r2 = aggregate_df[aggregate_df['Model'] == model.upper()]['R²'].values[0]
    pearson_r = aggregate_df[aggregate_df['Model'] == model.upper()]['Pearson r'].values[0]
    pearson_p = aggregate_df[aggregate_df['Model'] == model.upper()]['Pearson p'].values[0]
    slope_10k = aggregate_df[aggregate_df['Model'] == model.upper()]['Slope_per_10k'].values[0]

    sig = ""
    if pearson_p < 0.001:
        sig = "***"
    elif pearson_p < 0.01:
        sig = "**"
    elif pearson_p < 0.05:
        sig = "*"

    stats_text = f'r = {pearson_r:.3f}{sig}\nR² = {r2:.3f}\nSlope = {slope_10k:.4f} per $10k'
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
            verticalalignment='top', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax.set_xlabel('GDP per Capita (USD)', fontweight='bold', fontsize=11)
    ax.set_ylabel('Mean Absolute Error (pp)', fontweight='bold', fontsize=11)
    ax.set_title(f'{model.upper()}', fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)

    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}k'))

plt.tight_layout()
plt.savefig('gdp_aggregate_regression_clean.png', dpi=300, bbox_inches='tight')
plt.savefig('gdp_aggregate_regression_clean.pdf', dpi=300, bbox_inches='tight')
print("✓ Saved gdp_aggregate_regression_clean.png")
print("✓ Saved gdp_aggregate_regression_clean.pdf")
plt.close()

# Figure 1c: Aggregate scatter plots without labels (MAE vs log GDP)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Log GDP per Capita vs Prediction Error (Aggregated - No Labels)',
             fontsize=16, fontweight='bold')

for idx, model in enumerate(models):
    ax = axes[idx // 2, idx % 2]

    mae_col = f'mae_{model}'
    data = country_aggregate[[mae_col, 'log_gdp_capita_2021']].dropna()

    ax.scatter(data['log_gdp_capita_2021'], data[mae_col],
               alpha=0.6, s=60, c='steelblue', edgecolors='black', linewidth=0.5)

    X = data['log_gdp_capita_2021'].values.reshape(-1, 1)
    y = data[mae_col].values
    lr = LinearRegression()
    lr.fit(X, y)

    x_line = np.linspace(data['log_gdp_capita_2021'].min(), data['log_gdp_capita_2021'].max(), 100)
    y_line = lr.predict(x_line.reshape(-1, 1))

    ax.plot(x_line, y_line, 'r--', linewidth=2, label='Linear fit')

    # For simplicity, reuse the Pearson r and R² from level-GDP aggregate
    r2 = aggregate_df[aggregate_df['Model'] == model.upper()]['R²'].values[0]
    pearson_r = aggregate_df[aggregate_df['Model'] == model.upper()]['Pearson r'].values[0]
    pearson_p = aggregate_df[aggregate_df['Model'] == model.upper()]['Pearson p'].values[0]

    sig = ""
    if pearson_p < 0.001:
        sig = "***"
    elif pearson_p < 0.01:
        sig = "**"
    elif pearson_p < 0.05:
        sig = "*"

    stats_text = f'r (level GDP) = {pearson_r:.3f}{sig}\nR² (level GDP) = {r2:.3f}'
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
            verticalalignment='top', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax.set_xlabel('Log GDP per Capita (natural log)', fontweight='bold', fontsize=11)
    ax.set_ylabel('Mean Absolute Error (pp)', fontweight='bold', fontsize=11)
    ax.set_title(f'{model.upper()}', fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('log_gdp_aggregate_regression_clean.png', dpi=300, bbox_inches='tight')
plt.savefig('log_gdp_aggregate_regression_clean.pdf', dpi=300, bbox_inches='tight')
print("✓ Saved log_gdp_aggregate_regression_clean.png")
print("✓ Saved log_gdp_aggregate_regression_clean.pdf")
plt.close()

# Figure 2: By-stage heatmaps (Pearson r and slope per $10k)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('GDP per Capita vs MAE: Stage-by-Stage Analysis',
             fontsize=16, fontweight='bold')

# Panel A: Pearson r heatmap
ax = axes[0]
pivot_r = stage_df.pivot(index='Model', columns='Stage', values='Pearson r')
sns.heatmap(pivot_r, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'Pearson r'}, ax=ax, vmin=-1, vmax=1)
ax.set_title('(a) Pearson Correlation by Stage', fontweight='bold', fontsize=12)
ax.set_xlabel('Stage', fontweight='bold', fontsize=11)
ax.set_ylabel('Model', fontweight='bold', fontsize=11)

# Panel B: Slope heatmap
ax = axes[1]
pivot_slope = stage_df.pivot(index='Model', columns='Stage', values='Slope_per_10k')
sns.heatmap(pivot_slope, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'Slope (MAE change per $10k GDP)'}, ax=ax)
ax.set_title('(b) Regression Slope by Stage', fontweight='bold', fontsize=12)
ax.set_xlabel('Stage', fontweight='bold', fontsize=11)
ax.set_ylabel('Model', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('gdp_by_stage_heatmap.png', dpi=300, bbox_inches='tight')
plt.savefig('gdp_by_stage_heatmap.pdf', dpi=300, bbox_inches='tight')
print("✓ Saved gdp_by_stage_heatmap.png")
print("✓ Saved gdp_by_stage_heatmap.pdf")
plt.close()

# Figure 3: Stage-specific scatter plots with labels (for each model)
for model in models:
    fig, axes = plt.subplots(2, 4, figsize=(18, 10))
    fig.suptitle(f'{model.upper()}: GDP per Capita vs MAE by Stage (with Country Labels)',
                 fontsize=16, fontweight='bold')

    for stage_idx, stage in enumerate(sorted(df_complete['stage'].unique())):
        ax = axes[stage_idx // 4, stage_idx % 4]

        stage_data = df_complete[df_complete['stage'] == stage].copy()
        stage_country = stage_data.groupby('countrynew').agg({
            f'mae_{model}': 'mean',
            'gdp_capita_2021': 'first'
        }).reset_index()

        stage_country['country_code'] = stage_country['countrynew'].map(country_codes)

        data = stage_country[[f'mae_{model}', 'gdp_capita_2021', 'country_code']].dropna()

        if len(data) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                    transform=ax.transAxes)
            ax.set_title(f'Stage {stage}')
            continue

        ax.scatter(data['gdp_capita_2021'], data[f'mae_{model}'],
                   alpha=0.6, s=30, c='steelblue', edgecolors='black', linewidth=0.5)

        # Label a subset of countries (for readability)
        for idx2, row in data.iterrows():
            if idx2 % 5 == 0 or row[f'mae_{model}'] > data[f'mae_{model}'].quantile(0.9) or \
               row[f'mae_{model}'] < data[f'mae_{model}'].quantile(0.1):
                ax.text(row['gdp_capita_2021'], row[f'mae_{model}'],
                        row['country_code'], fontsize=5, alpha=0.6)

        X = data['gdp_capita_2021'].values.reshape(-1, 1)
        y = data[f'mae_{model}'].values

        lr = LinearRegression()
        lr.fit(X, y)

        x_line = np.linspace(data['gdp_capita_2021'].min(), data['gdp_capita_2021'].max(), 100)
        y_line = lr.predict(x_line.reshape(-1, 1))

        ax.plot(x_line, y_line, 'r--', linewidth=1.5)

        stage_stats = stage_df[(stage_df['Stage'] == stage) &
                               (stage_df['Model'] == model.upper())]
        if len(stage_stats) > 0:
            r = stage_stats['Pearson r'].values[0]
            p = stage_stats['Pearson p'].values[0]
            slope_10k = stage_stats['Slope_per_10k'].values[0]

            sig = ""
            if p < 0.001:
                sig = "***"
            elif p < 0.01:
                sig = "**"
            elif p < 0.05:
                sig = "*"

            ax.text(0.05, 0.95, f'r={r:.2f}{sig}\nslope={slope_10k:.3f} per $10k',
                    transform=ax.transAxes, verticalalignment='top', fontsize=8,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

        ax.set_title(f'Stage {stage}', fontweight='bold', fontsize=10)
        ax.set_xlabel('GDP per Capita (USD)', fontsize=9)
        ax.set_ylabel('MAE (pp)', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}k'))

    plt.tight_layout()
    plt.savefig(f'gdp_by_stage_{model}_labeled.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'gdp_by_stage_{model}_labeled.pdf', dpi=300, bbox_inches='tight')
    print(f"✓ Saved gdp_by_stage_{model}_labeled.png")
    print(f"✓ Saved gdp_by_stage_{model}_labeled.pdf")
    plt.close()

# Figure 4: Stage-specific scatter plots WITHOUT labels (cleaner)
for model in models:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f'{model.upper()}: GDP per Capita vs MAE by Stage',
                 fontsize=16, fontweight='bold')

    for stage_idx, stage in enumerate(sorted(df_complete['stage'].unique())):
        ax = axes[stage_idx // 4, stage_idx % 4]

        stage_data = df_complete[df_complete['stage'] == stage].copy()
        stage_country = stage_data.groupby('countrynew').agg({
            f'mae_{model}': 'mean',
            'gdp_capita_2021': 'first'
        }).reset_index()

        data = stage_country[[f'mae_{model}', 'gdp_capita_2021']].dropna()

        if len(data) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                    transform=ax.transAxes)
            ax.set_title(f'Stage {stage}')
            continue

        ax.scatter(data['gdp_capita_2021'], data[f'mae_{model}'],
                   alpha=0.6, s=35, c='steelblue', edgecolors='black', linewidth=0.5)

        X = data['gdp_capita_2021'].values.reshape(-1, 1)
        y = data[f'mae_{model}'].values

        lr = LinearRegression()
        lr.fit(X, y)

        x_line = np.linspace(data['gdp_capita_2021'].min(), data['gdp_capita_2021'].max(), 100)
        y_line = lr.predict(x_line.reshape(-1, 1))

        ax.plot(x_line, y_line, 'r--', linewidth=2)

        stage_stats = stage_df[(stage_df['Stage'] == stage) &
                               (stage_df['Model'] == model.upper())]
        if len(stage_stats) > 0:
            r = stage_stats['Pearson r'].values[0]
            p = stage_stats['Pearson p'].values[0]
            slope_10k = stage_stats['Slope_per_10k'].values[0]

            sig = ""
            if p < 0.001:
                sig = "***"
            elif p < 0.01:
                sig = "**"
            elif p < 0.05:
                sig = "*"

            ax.text(0.05, 0.95, f'r={r:.2f}{sig}\nslope={slope_10k:.3f} per $10k',
                    transform=ax.transAxes, verticalalignment='top', fontsize=9,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

        ax.set_title(f'Stage {stage}', fontweight='bold', fontsize=11)
        ax.set_xlabel('GDP per Capita (USD)', fontsize=9)
        ax.set_ylabel('MAE (pp)', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}k'))

    plt.tight_layout()
    plt.savefig(f'gdp_by_stage_{model}_clean.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'gdp_by_stage_{model}_clean.pdf', dpi=300, bbox_inches='tight')
    print(f"✓ Saved gdp_by_stage_{model}_clean.png")
    print(f"✓ Saved gdp_by_stage_{model}_clean.pdf")
    plt.close()

# ================================================================
# 7. Export Results
# ================================================================

print("\n" + "="*80)
print("7. EXPORTING RESULTS")
print("="*80)

# Save aggregate results
aggregate_df.to_csv('gdp_aggregate_regression.csv', index=False)
print("✓ Saved gdp_aggregate_regression.csv")

# Save by-stage results
stage_df.to_csv('gdp_by_stage_regression.csv', index=False)
print("✓ Saved gdp_by_stage_regression.csv")

# Save full country-stage dataset (for reproducibility)
df_complete.to_csv('gdp_full_country_stage_dataset.csv', index=False)
print("✓ Saved gdp_full_country_stage_dataset.csv")

# ================================================================
# 8. Summary
# ================================================================

print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print("\n1. AGGREGATE RESULTS (Across All Stages):")
print("-" * 60)
for _, row in aggregate_df.iterrows():
    sig = ""
    if row['Pearson p'] < 0.001:
        sig = "***"
    elif row['Pearson p'] < 0.01:
        sig = "**"
    elif row['Pearson p'] < 0.05:
        sig = "*"

    direction = "NEGATIVE" if row['Slope_per_10k'] < 0 else "POSITIVE"

    print(f"\n{row['Model']}:")
    print(f"   {direction} relationship: slope = {row['Slope_per_10k']:.4f} MAE per $10k")
    print(f"   Correlation: r = {row['Pearson r']:.3f} (p = {row['Pearson p']:.4f}) {sig}")
    print(f"   Variance explained: R² = {row['R²']:.3f}")

    if row['Slope_per_10k'] < 0:
        print(f"   → Higher GDP per capita = LOWER prediction error")
    else:
        print(f"   → Higher GDP per capita = HIGHER prediction error")

print("\n\n2. BY-STAGE PATTERNS:")
print("-" * 60)

for model in models:
    model_stages = stage_df[stage_df['Model'] == model.upper()].sort_values('Stage')

    negative_stages = model_stages[model_stages['Slope_per_10k'] < 0]['Stage'].tolist()
    positive_stages = model_stages[model_stages['Slope_per_10k'] > 0]['Stage'].tolist()
    sig_stages = model_stages[model_stages['Pearson p'] < 0.05]['Stage'].tolist()

    print(f"\n{model.upper()}:")
    if negative_stages:
        print(f"   Negative relationship (stages): {negative_stages}")
    if positive_stages:
        print(f"   Positive relationship (stages): {positive_stages}")
    if sig_stages:
        print(f"   Significant at p<.05 (stages): {sig_stages}")
    else:
        print(f"   No significant relationships")

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)

print("\nFiles created:")
print("  - gdp_aggregate_regression.csv")
print("  - gdp_by_stage_regression.csv")
print("  - gdp_full_country_stage_dataset.csv")
print("  - gdp_aggregate_regression_with_labels.png / .pdf")
print("  - gdp_aggregate_regression_clean.png / .pdf")
print("  - log_gdp_aggregate_regression_clean.png / .pdf")
print("  - gdp_by_stage_heatmap.png / .pdf")
print("  - gdp_by_stage_gpt_labeled.png / .pdf")
print("  - gdp_by_stage_gpt_clean.png / .pdf")
print("  - gdp_by_stage_claude_labeled.png / .pdf")
print("  - gdp_by_stage_claude_clean.png / .pdf")
print("  - gdp_by_stage_gemini_labeled.png / .pdf")
print("  - gdp_by_stage_gemini_clean.png / .pdf")
print("  - gdp_by_stage_llama_labeled.png / .pdf")
print("  - gdp_by_stage_llama_clean.png / .pdf")
print("\nTotal: 3 CSV files + multiple image files (PNG + PDF)")


⚠️  adjustText not installed - country labels may overlap
   Install with: pip install adjustText
GDP PER CAPITA vs PREDICTION ERROR ANALYSIS

1. LOADING DATA
✓ Loaded predictions: 1000 rows
✓ Merged ground truth and GDP

✓ Final dataset: 1000 rows
   Countries: 125
   GDP per capita range: $2,482 - $123,671

Created log GDP per capita (natural log).
   Mean log GDP: 9.87

2. CALCULATING MAE
✓ Calculated MAE for all models

3. AGGREGATE REGRESSION (Across All Stages)

✓ Aggregated data: 125 countries
   GDP per capita (min/median/max): $2,482 / $19,159 / $123,671

GPT:
   N countries: 125
   Slope: -0.6429 (MAE change per $10k GDP per capita)
   Intercept: 12.42
   R²: 0.1093
   Pearson r: -0.3306, p = 0.0002
   Spearman ρ: -0.4727, p = 0.0000

CLAUDE:
   N countries: 125
   Slope: -0.8957 (MAE change per $10k GDP per capita)
   Intercept: 10.52
   R²: 0.1474
   Pearson r: -0.3840, p = 0.0000
   Spearman ρ: -0.5329, p = 0.0000

GEMINI:
   N countries: 125
   Slope: -1.3212 (MAE change 

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>